In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
# Generate random non-overlapping rectangles in the plane.
np.random.seed(0) # fixed seed for random number generator
m = 30 # number of rectangles
L = np.zeros((m, 2)) # lower left corners of rectangles
U = np.zeros((m, 2)) # upper right corners of rectangles
for i in range(m):
    while True: # loop until random box does not overlap with previous
        ci = np.random.uniform(0, 1, 2) # center
        d = np.abs(np.random.randn(2) / 10) # half diagonal
        L[i] = ci - d
        U[i] = ci + d
        overlap = False
        for j in range(i): # check if random box overlaps with previous
            lij = np.maximum(L[i], L[j]) # lower left corner of intersection
            uij = np.minimum(U[i], U[j]) # upper right corner of intersection
            if np.all(lij <= uij): # checks nonemptiness of intersection
                overlap = True
                break
        if not overlap:
            break

In [ ]:
# Start point is center of bottom left box.
i = np.argmin(np.sum(L, axis=1))
x0 = (L[i] + U[i]) / 2

# Goal point is center of top right box.
i = np.argmax(np.sum(U, axis=1))
xK = (L[i] + U[i]) / 2

In [ ]:
# Footstep planning problem.
K = 10 # number of footsteps
x = cp.Variable((K + 1, 2)) # footstep locations

# Initial and final positions.
constraints = [x[0] == x0, x[-1] == xK]

# Disjunctive constraints.
for k in range(1, K):
    xk = cp.Variable((m, 2)) # auxiliary continuous variables
    yk = cp.Variable(m, boolean=True) # auxiliary binary variables
    constraints.append(cp.sum(xk, axis=0) == x[k])
    constraints.append(cp.sum(yk) == 1)
    for i in range(m): # homogenization constraints
        constraints.append(xk[i] >= L[i] * yk[i])
        constraints.append(xk[i] <= U[i] * yk[i])

# Objective function.
obj = cp.sum_squares(x[1:] - x[:-1])

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(solver='GUROBI')

In [ ]:
# Plot solution.
plt.figure()
plt.gca().set_aspect('equal')
for li, ui in zip(L, U):
    rect = Rectangle(li, *(ui - li), ec='black', fc='mintcream')
    plt.gca().add_patch(rect)
plt.plot(*x.value.T, marker='o', ls='--')
plt.xlim(-.2, 1.2)
plt.ylim(-.2, 1.2)
plt.show()